# 01 — Ingestion dans le data lake : couche *bronze*

Consommation électrique Enedis (1/2 h, par région, 2022) et température quotidienne départementale (ODRÉ).
Les données brutes sont déposées **telles quelles** dans RustFS, au format Delta.
Seules modifications autorisées : les noms de colonnes (caractères interdits par Delta) et deux colonnes techniques
(`_source`, `_ingere_le`).

```
data/raw/enedis/*.csv       ──►  s3a://datalake/bronze/enedis_conso
data/raw/meteo/*.csv        ──►  s3a://datalake/bronze/meteo_departements
ref/departements_regions.csv ──►  s3a://datalake/bronze/ref_departements
```

Prérequis : `bash scripts/telecharger_donnees.sh` puis `docker compose up -d`.

In [ ]:
import sys; sys.path.append("/home/jovyan/work")
import glob, re, unicodedata
from pyspark.sql import functions as F
from commun.spark_utils import creer_session, creer_bucket, format_tables, RAW, LAKE, WORK

creer_bucket("datalake")

In [ ]:
spark = creer_session("bronze")
FMT = format_tables(spark)
FMT

## 1. Fonctions utilitaires

In [ ]:
def detecter_sep(chemin):
    """Les CSV français utilisent souvent « ; » : on le détecte sur la première ligne."""
    with open(chemin, encoding="utf-8-sig") as f:
        ligne = f.readline()
    return ";" if ligne.count(";") >= ligne.count(",") else ","


def nettoyer_noms(df):
    """Delta refuse certains caractères dans les noms de colonnes : on ne renomme que ceux-là."""
    for c in df.columns:
        propre = unicodedata.normalize("NFKD", c.strip().lower()).encode("ascii", "ignore").decode()
        propre = re.sub(r"[ ,;{}()\n\t=]+", "_", propre)
        if propre != c:
            df = df.withColumnRenamed(c, propre)
    return df


def vers_bronze(df, nom):
    """Écrit en bronze avec traçabilité (fichier source, date d'ingestion). Aucune autre transformation."""
    df = (nettoyer_noms(df)
          .withColumn("_source", F.col("_metadata.file_name"))
          .withColumn("_ingere_le", F.current_timestamp()))
    df.write.format(FMT).mode("overwrite").save(f"{LAKE}/bronze/{nom}")
    relu = spark.read.format(FMT).load(f"{LAKE}/bronze/{nom}")
    print(f"bronze/{nom} : {relu.count():,} lignes, {len(relu.columns)} colonnes")
    return relu

## 2. Consommation Enedis → bronze

Tout est lu en **texte** (`inferSchema=False`) : le typage est le travail de la couche *silver*.

In [ ]:
fichiers = sorted(glob.glob(f"{RAW}/enedis/*.csv"))
assert fichiers, "Aucun CSV Enedis : lancez d'abord  bash scripts/telecharger_donnees.sh"
sep = detecter_sep(fichiers[0])
print(len(fichiers), "fichiers, séparateur", repr(sep))

conso_raw = spark.read.csv(fichiers, header=True, sep=sep, inferSchema=False)
conso = vers_bronze(conso_raw, "enedis_conso")
conso.printSchema()
conso.show(5, truncate=False)

In [ ]:
# Contrôles : volume attendu (> 2 millions de lignes) et répartition par fichier source
n = conso.count()
assert n > 2_000_000, f"Seulement {n:,} lignes : vérifier le téléchargement"
conso.groupBy("_source").count().orderBy("_source").show(truncate=False)

## 3. Températures ODRÉ → bronze

In [ ]:
fichier_meteo = f"{RAW}/meteo/temperature_departementale.csv"
meteo_raw = spark.read.csv(fichier_meteo, header=True, sep=detecter_sep(fichier_meteo), inferSchema=False)
meteo = vers_bronze(meteo_raw, "meteo_departements")
meteo.printSchema()
meteo.show(5, truncate=False)

## 4. Référentiel départements → régions

Petit fichier statique versionné dans `ref/` (96 départements métropolitains) : il permet de passer de la maille
départementale (météo) à la maille régionale (Enedis) en couche *silver*.

In [ ]:
ref_raw = spark.read.csv(f"{WORK}/ref/departements_regions.csv", header=True, sep=";", inferSchema=False)
ref = vers_bronze(ref_raw, "ref_departements")
ref.show(5)

## 5. Historique Delta

In [ ]:
if FMT == "delta":
    from delta.tables import DeltaTable
    for t in ["enedis_conso", "meteo_departements", "ref_departements"]:
        print(t)
        DeltaTable.forPath(spark, f"{LAKE}/bronze/{t}").history().select("version", "timestamp", "operation").show(truncate=False)

## C'est réussi quand…
- la console RustFS (http://localhost:9001) montre `bronze/enedis_conso`, `bronze/meteo_departements`, `bronze/ref_departements` ;
- `bronze/enedis_conso` dépasse 2 millions de lignes ;
- les noms de colonnes des deux jeux sont notés pour écrire les règles *silver* (notebook 02).

In [ ]:
spark.stop()